# Matched Sobolev smoothing control (50D)

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.4, Table 4, Appendix B.2.

Compares Gaeta–Lie with a state-Jacobian (Sobolev) penalty under identical Stage-I checkpoints, minibatches, training budgets and 256 collocation points, with gradient-matched weights tuned on validation NLL. Also includes data-only continuation and random-W controls.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, global knobs for E2

In [ ]:
# ============================================================
# Cell 1 — Config, imports, global knobs for E2
# ============================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import csv
import io
import json
import math
import os
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


CONFIG = {
    # Reproducibility
    "SEED": 42,

    # ----------------------------
    # 50D synthetic SDE generation
    # ----------------------------
    "STATE_DIM": 50,

    # Primary E2 paper systems:
    #   "ring_double_well_50d"
    #   "low_rank_ou_50d"
    #   "tanh_chain_50d"
    # Optional registry system:
    #   "block_lorenz_50d"
    "TRUE_SDE_NAME": "ring_double_well_50d",

    "T_FINAL": 2.0,
    "DT": 0.01,
    "N_TRAJECTORIES": 64,
    "X0_MEAN": 0.0,
    "X0_STD": 1.0,
    "TRAIN_FRACTION": 0.8,

    # ----------------------------
    # Neural SDE surrogate
    # ----------------------------
    "MLP_HIDDEN_DIM": 192,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 3.0,
    "SIGMA_EPS": 1e-4,

    # ----------------------------
    # Stage I: baseline surrogate
    # ----------------------------
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # ----------------------------
    # Stage II: symmetry discovery
    # ----------------------------
    "N_GENERATORS": 50,
    "SYM_DISCOVERY_EPOCHS": 4000,
    "SYM_DISCOVERY_LR": 5e-3,
    "SYM_DISCOVERY_POINTS": 256,
    "SYM_DISCOVERY_T_BOX": (0.0, 2.0),
    "SYM_DISCOVERY_X_BOX": (-3.0, 3.0),
    "SYM_INIT_SCALE": 0.03,

    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.5,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    "BASIS_MODE": "expanded_gaeta_lowrank",
    "BASIS_LINEAR_MU_LIST": [-1.0, -0.5, 0.5, 1.0],
    "BASIS_EXP_CLIP": 6.0,
    "BASIS_LOW_RANK_K": 6,
    "BASIS_ADD_EXPONENTIAL": True,
    "BASIS_ADD_LOW_RANK_ROTATIONS": True,
    "BASIS_ADD_LOW_RANK_SHEARS": True,

    # ========================================================
    # E2: matched Stage-III controls
    # ========================================================
    "E2_ARMS": (
        "data_only",
        "random_w",
        "sobolev",
        "gaeta_lie",
    ),

    # Final matched Stage-III budget.
    "E2_STAGE3_EPOCHS": 800,
    "E2_STAGE3_BATCH_SIZE": 512,
    "E2_STAGE3_LR": 3e-4,

    # Every regularized arm receives exactly the same number and
    # locations of trajectory and extra collocation points.
    "E2_N_DATA_REG_POINTS": 128,
    "E2_N_EXTRA_REG_POINTS": 128,
    "E2_JITTER_STD": 0.05,

    # The submitted 50D Gaeta-Lie weight is the reference weighted
    # gradient strength. Random-W and Sobolev base weights are matched
    # to it at the common Stage-I checkpoint.
    "E2_GL_REFERENCE_LAMBDA": 105000.0,
    "E2_WEIGHT_MULTIPLIERS": (0.3, 1.0, 3.0),

    # Weight tuning uses only a Stage-III fit/validation split carved
    # from the original training transitions. Ground-truth coefficient
    # error and the original test set are not used for selection.
    "E2_TUNING_EPOCHS": 150,
    "E2_STAGE3_VALIDATION_FRACTION": 0.10,

    # One Rademacher state-space probe per collocation point gives an
    # unbiased Hutchinson estimate of each Jacobian Frobenius norm.
    "E2_SOB_HUTCHINSON_PROBES": 1,

    # Independent deterministic streams.
    "E2_RANDOM_W_SEED": 271828,
    "E2_CALIBRATION_SEED": 161803,
    "E2_TUNING_SCHEDULE_SEED": 57721,
    "E2_FINAL_SCHEDULE_SEED": 141421,
    "E2_EVAL_SEED": 314159,

    # Final evaluation sizes.
    "E2_N_COEFF_EVAL_POINTS": 2000,
    "E2_N_RESIDUAL_EVAL_POINTS": 256,
    "E2_N_EXACT_SOB_EVAL_POINTS": 32,
    "E2_MATERIAL_ADVANTAGE_PCT": 2.0,

    # Coefficient MSE is evaluated on the same uniform time-state box
    # convention used by the manuscript.
    "E2_COEFF_EVAL_T_BOX": (0.0, 2.0),
    "E2_COEFF_EVAL_X_BOX": (-3.0, 3.0),

    # Diagnostics/plots.
    "N_EVAL_POINTS": 2000,
    "N_PLOT_GRID": 160,
    "PLOT_COMPONENTS": [0, 1, 2, 9, 24, 49],
}

D = CONFIG["STATE_DIM"]
Q = D + 1
key = jax.random.PRNGKey(CONFIG["SEED"])

## Cell 2 — Editable 50D SDE registry and synthetic data

In [ ]:
# ============================================================
# Cell 2 — Editable 50D SDE registry and synthetic data
# ============================================================

# ------------------------------------------------------------------
# How to use:
#
# Change CONFIG["TRUE_SDE_NAME"] in Cell 1.
#
# Available examples:
#   "ring_double_well_50d"
#   "low_rank_ou_50d"
#   "block_lorenz_50d"
#   "tanh_chain_50d"
#
# Each SDE entry defines:
#   drift_fn(t, x) -> shape (..., 50)
#   sigma_fn(t, x) -> shape (..., 50), positive diagonal diffusion
# ------------------------------------------------------------------


def _idx_float():
    return jnp.arange(D, dtype=jnp.float32)


def _phase_vector():
    return 2.0 * jnp.pi * _idx_float() / float(D)


def _mode_sin(k):
    return jnp.sin(2.0 * jnp.pi * k * _idx_float() / float(D))


def _mode_cos(k):
    return jnp.cos(2.0 * jnp.pi * k * _idx_float() / float(D))


def _normalize_mode(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


# ============================================================
# 50D SDE example 1: ring-coupled double-well
# ============================================================

def drift_ring_double_well_50d(t, x):
    """
    50D nonlinear ring-coupled double-well SDE.

    Drift:
        f_i = x_i - x_i^3
              + c (x_{i-1} + x_{i+1} - 2 x_i)
              + weak time forcing_i(t)

    Nontrivial features:
        - nonlinear bistability in every coordinate
        - local spatial coupling on a ring
        - weak heterogeneous forcing
    """
    c = 0.25
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x

    phase = _phase_vector()
    forcing = 0.08 * jnp.sin(2.0 * jnp.pi * t + phase)

    return x - x**3 + c * lap + forcing


def sigma_ring_double_well_50d(t, x):
    """
    Positive state-dependent diagonal diffusion.
    """
    phase = _phase_vector()
    base = 0.18 + 0.02 * jnp.sin(phase) ** 2
    return base + 0.04 * jax.nn.sigmoid(x)


# ============================================================
# 50D SDE example 2: low-rank coupled OU/spiral system
# ============================================================

def drift_low_rank_ou_50d(t, x):
    """
    Stable 50D linear/nonlinear low-rank OU-like SDE.

    Drift:
        stable coordinate damping
        + skew ring coupling
        + low-rank global coupling
        + weak periodic forcing
        + small tanh nonlinearity

    This is useful because the true drift has global low-rank structure
    but is still learned by a generic neural SDE.
    """
    idx = _idx_float()

    # Coordinate-dependent damping
    lam = 0.45 + 0.30 * (idx / float(D - 1))

    # Ring skew/transport-like coupling
    skew = 0.18 * (jnp.roll(x, -1, axis=-1) - jnp.roll(x, 1, axis=-1))

    v1 = _normalize_mode(_mode_sin(1))
    v2 = _normalize_mode(_mode_cos(2))
    v3 = _normalize_mode(_mode_sin(3))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)
    p3 = jnp.sum(x * v3, axis=-1, keepdims=True)

    low_rank = (
        0.45 * p1 * v2
        - 0.35 * p2 * v1
        + 0.25 * jnp.tanh(p3) * v3
    )

    forcing = 0.05 * (
        jnp.sin(1.2 * t) * v1
        + jnp.cos(0.8 * t) * v2
    )

    return -lam * x + skew + low_rank + forcing + 0.05 * jnp.tanh(x)


def sigma_low_rank_ou_50d(t, x):
    v1 = _normalize_mode(_mode_sin(1))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    base = 0.20 + 0.03 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jax.nn.sigmoid(p1) + 0.02 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 3: block Lorenz-style system
# ============================================================

def drift_block_lorenz_50d(t, x):
    """
    50D block-tamed Lorenz-style SDE.

    First 48 coordinates form 16 weakly coupled Lorenz-like triples.
    Last 2 coordinates are stable auxiliary variables coupled to block means.

    The Lorenz blocks are tamed to avoid numerical explosions.
    """
    x_blocks = x[..., :48].reshape(x.shape[:-1] + (16, 3))
    y = x[..., 48:50]

    x1 = x_blocks[..., 0]
    x2 = x_blocks[..., 1]
    x3 = x_blocks[..., 2]

    sigma_l = 4.0
    rho = 8.0
    beta = 8.0 / 3.0

    g1 = sigma_l * (x2 - x1)
    g2 = x1 * (rho - x3) - x2
    g3 = x1 * x2 - beta * x3

    raw = jnp.stack([g1, g2, g3], axis=-1)

    norm2 = jnp.sum(x_blocks**2, axis=-1, keepdims=True)
    tame = 1.0 + 0.05 * norm2

    block_f = raw / tame

    # Weak coupling between neighboring blocks
    block_lap = (
        jnp.roll(x_blocks, 1, axis=-2)
        + jnp.roll(x_blocks, -1, axis=-2)
        - 2.0 * x_blocks
    )
    block_f = block_f + 0.05 * block_lap

    # Auxiliary coordinates
    block_mean = jnp.mean(x_blocks, axis=(-2, -1), keepdims=False)
    f_y1 = -0.60 * y[..., 0] + 0.10 * jnp.tanh(block_mean)
    f_y2 = -0.75 * y[..., 1] + 0.10 * jnp.sin(t)

    f_y = jnp.stack([f_y1, f_y2], axis=-1)

    return jnp.concatenate([block_f.reshape(x.shape[:-1] + (48,)), f_y], axis=-1)


def sigma_block_lorenz_50d(t, x):
    base = 0.12 + 0.02 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 4: nonlinear tanh reaction chain
# ============================================================

def drift_tanh_chain_50d(t, x):
    """
    50D smooth nonlinear reaction-chain SDE.

    Drift:
        damping
        + forward/backward tanh couplings
        + weak global low-rank feedback
        + heterogeneous forcing

    This is bounded and stable, but still nonlinear and coupled.
    """
    phase = _phase_vector()

    f_local = (
        -0.55 * x
        + 0.35 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.25 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.10 * jnp.tanh(jnp.roll(x, -2, axis=-1))
    )

    v1 = _normalize_mode(_mode_cos(1))
    v2 = _normalize_mode(_mode_sin(2))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)

    global_feedback = 0.20 * jnp.tanh(p1) * v2 - 0.15 * jnp.tanh(p2) * v1

    forcing = 0.04 * jnp.sin(1.5 * t + phase)

    return f_local + global_feedback + forcing


def sigma_tanh_chain_50d(t, x):
    return 0.22 + 0.04 * jnp.sin(x) ** 2


# ============================================================
# Registry
# ============================================================

SDE_EXAMPLES = {
    "ring_double_well_50d": {
        "description": "50D nonlinear ring-coupled double-well SDE with state-dependent diffusion.",
        "drift_fn": drift_ring_double_well_50d,
        "sigma_fn": sigma_ring_double_well_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "input_x_scale": 2.5,
        "x_box": (-2.5, 2.5),
    },

    "low_rank_ou_50d": {
        "description": "50D stable low-rank OU/spiral SDE with global modes and state-dependent diffusion.",
        "drift_fn": drift_low_rank_ou_50d,
        "sigma_fn": sigma_low_rank_ou_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },

    "block_lorenz_50d": {
        "description": "50D tamed Lorenz-block SDE: 16 coupled triples plus 2 stable auxiliary coordinates.",
        "drift_fn": drift_block_lorenz_50d,
        "sigma_fn": sigma_block_lorenz_50d,
        "t_final": 1.0,
        "dt": 0.005,
        "x0_mean": 0.0,
        "x0_std": 0.8,
        "input_x_scale": 5.0,
        "x_box": (-5.0, 8.0),
    },

    "tanh_chain_50d": {
        "description": "50D damped nonlinear tanh reaction chain with low-rank feedback.",
        "drift_fn": drift_tanh_chain_50d,
        "sigma_fn": sigma_tanh_chain_50d,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
}


def activate_sde_from_config():
    """
    Select active SDE from CONFIG["TRUE_SDE_NAME"] and update
    dependent CONFIG fields automatically.
    """
    name = CONFIG["TRUE_SDE_NAME"]

    if name not in SDE_EXAMPLES:
        raise ValueError(
            f"Unknown TRUE_SDE_NAME={name}. "
            f"Available choices: {list(SDE_EXAMPLES.keys())}"
        )

    spec = SDE_EXAMPLES[name]

    CONFIG["T_FINAL"] = float(spec["t_final"])
    CONFIG["DT"] = float(spec["dt"])
    CONFIG["X0_MEAN"] = spec["x0_mean"]
    CONFIG["X0_STD"] = spec["x0_std"]
    CONFIG["INPUT_T_SCALE"] = float(spec["t_final"])
    CONFIG["INPUT_X_SCALE"] = float(spec["input_x_scale"])

    x_box = tuple(float(v) for v in spec["x_box"])

    CONFIG["SYM_DISCOVERY_T_BOX"] = (0.0, float(spec["t_final"]))
    CONFIG["STAGE_F_EXTRA_T_BOX"] = (0.0, float(spec["t_final"]))

    CONFIG["SYM_DISCOVERY_X_BOX"] = x_box
    CONFIG["STAGE_F_EXTRA_X_BOX"] = x_box

    return spec


ACTIVE_SDE = activate_sde_from_config()


def _as_50d_vector(value, name):
    """
    Convert scalar or length-D array/tuple into shape (D,).
    """
    arr = jnp.asarray(value, dtype=jnp.float32)

    if arr.ndim == 0:
        return jnp.full((D,), arr, dtype=jnp.float32)

    if arr.shape != (D,):
        raise ValueError(f"{name} must be scalar or shape ({D},), got {arr.shape}")

    return arr


def true_drift(t, x):
    """
    Dispatch to active 50D drift.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    return ACTIVE_SDE["drift_fn"](t, x)


def true_sigma(t, x):
    """
    Dispatch to active 50D diagonal diffusion.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    sigma = ACTIVE_SDE["sigma_fn"](t, x)
    return jnp.maximum(sigma, 1e-6)


def print_active_sde_report():
    print("\n" + "=" * 72)
    print("ACTIVE 50D SYNTHETIC SDE")
    print("=" * 72)
    print(f"TRUE_SDE_NAME: {CONFIG['TRUE_SDE_NAME']}")
    print(f"Description  : {ACTIVE_SDE['description']}")
    print(f"STATE_DIM    : {D}")
    print(f"T_FINAL      : {CONFIG['T_FINAL']}")
    print(f"DT           : {CONFIG['DT']}")
    print(f"X0_MEAN      : {CONFIG['X0_MEAN']}")
    print(f"X0_STD       : {CONFIG['X0_STD']}")
    print(f"X_BOX        : {CONFIG['STAGE_F_EXTRA_X_BOX']}")
    print(f"INPUT_T_SCALE: {CONFIG['INPUT_T_SCALE']}")
    print(f"INPUT_X_SCALE: {CONFIG['INPUT_X_SCALE']}")
    print("=" * 72)


print_active_sde_report()


@jax.jit
def simulate_sde_paths(key):
    """
    Simulate all trajectories using Euler-Maruyama.

    Returns
    -------
    times : shape (n_steps + 1,)
    paths : shape (n_paths, n_steps + 1, 50)
    """
    T = CONFIG["T_FINAL"]
    dt = CONFIG["DT"]
    n_steps = int(round(T / dt))
    n_paths = CONFIG["N_TRAJECTORIES"]

    key_x0, key_noise = jax.random.split(key)

    x0_mean = _as_50d_vector(CONFIG["X0_MEAN"], "X0_MEAN")
    x0_std = _as_50d_vector(CONFIG["X0_STD"], "X0_STD")

    x0 = x0_mean[None, :] + x0_std[None, :] * jax.random.normal(
        key_x0,
        (n_paths, D),
    )

    noise = jax.random.normal(key_noise, (n_steps, n_paths, D))

    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, zeta = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * zeta
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], noise))

    paths = jnp.concatenate([x0[None, :, :], xs_tail], axis=0)
    paths = jnp.transpose(paths, (1, 0, 2))

    return times, paths


def make_increment_dataset(times, paths):
    """
    Convert simulated trajectories into one-step increment data.

    Returns:
        t0 : shape (n_samples,)
        x0 : shape (n_samples, 50)
        dt : shape (n_samples,)
        dx : shape (n_samples, 50)
    """
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    dt = CONFIG["DT"]

    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt_arr = jnp.full((x0.shape[0],), dt, dtype=jnp.float32)

    return t0, x0, dt_arr, dx


def train_test_split_dataset(key, dataset, train_fraction):
    n = dataset[0].shape[0]
    perm = jax.random.permutation(key, n)
    n_train = int(train_fraction * n)

    idx_train = perm[:n_train]
    idx_test = perm[n_train:]

    train = tuple(arr[idx_train] for arr in dataset)
    test = tuple(arr[idx_test] for arr in dataset)

    return train, test


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


# ----------------------------
# Generate data
# ----------------------------
key, subkey = jax.random.split(key)
times, paths = simulate_sde_paths(subkey)
dataset = make_increment_dataset(times, paths)

key, subkey = jax.random.split(key)
train_data, test_data = train_test_split_dataset(
    subkey,
    dataset,
    CONFIG["TRAIN_FRACTION"],
)

print("times shape:", times.shape)
print("paths shape:", paths.shape)
print("train samples:", train_data[0].shape[0])
print("test samples:", test_data[0].shape[0])


# ----------------------------
# Quick trajectory plots for selected components only
# ----------------------------
for comp in CONFIG["PLOT_COMPONENTS"]:
    plt.figure(figsize=(8, 4))
    for i in range(min(12, CONFIG["N_TRAJECTORIES"])):
        plt.plot(np.array(times), np.array(paths[i, :, comp]), alpha=0.6)
    plt.xlabel("t")
    plt.ylabel(f"X_{comp+1}(t)")
    plt.title(
        f"{CONFIG['TRUE_SDE_NAME']} trajectories: component {comp+1}"
    )
    plt.grid(True)
    plt.show()


# ----------------------------
# Sanity check drift/diffusion values
# ----------------------------
key, subkey = jax.random.split(key)
x_probe = jax.random.normal(subkey, (8, D))
t_probe = 0.5 * CONFIG["T_FINAL"]

print("\nDrift probe shape:", true_drift(t_probe, x_probe).shape)
print("Sigma probe shape:", true_sigma(t_probe, x_probe).shape)
print("Sigma probe min  :", float(jnp.min(true_sigma(t_probe, x_probe))))

## Cell 3 — 50D Neural SDE surrogate, Adam optimizer, baseline training

In [ ]:
# ============================================================
# Cell 3 — 50D Neural SDE surrogate, Adam optimizer, baseline training
# ============================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    """
    Small pure-JAX MLP class.

    For the 3D SDE:
        input  = (t, x1, x2, x3), dimension 4
        output = (f1, f2, f3, raw_sigma1, raw_sigma2, raw_sigma3), dimension 6
    """

    def __init__(self, in_dim=4, hidden_dim=96, depth=3, out_dim=6):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.out_dim = out_dim

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)

        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            W = glorot_uniform(k, in_d, out_d)
            b = jnp.zeros((out_d,), dtype=jnp.float32)
            params.append({"W": W, "b": b})
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        out = h @ params[-1]["W"] + params[-1]["b"]
        return out


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x):
    tn = jnp.asarray([t / CONFIG["INPUT_T_SCALE"]], dtype=jnp.float32)
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.concatenate([tn, xn], axis=0)


def surrogate_f_sigma(params, t, x):
    z = normalize_input(t, x)
    raw = surrogate_model.apply(params, z)

    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_vec_single(params, z):
    """
    z = (t, x1, x2, x3)
    returns f(t,x), shape (3,)
    """
    return surrogate_f_sigma(params, z[0], z[1:])[0]


def sigma_vec_single(params, z):
    """
    z = (t, x1, x2, x3)
    returns diagonal sigma(t,x), shape (3,)
    """
    return surrogate_f_sigma(params, z[0], z[1:])[1]


def data_nll_loss(params, batch):
    """
    Euler-Maruyama diagonal-Gaussian negative log-likelihood.

    batch = (t0, x0, dt, dx)
        t0 shape (B,)
        x0 shape (B, 3)
        dt shape (B,)
        dx shape (B, 3)
    """
    t0, x0, dt, dx = batch

    f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)

    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1e-8

    nll_components = 0.5 * (
        (dx - mean) ** 2 / var
        + jnp.log(2.0 * jnp.pi * var)
    )

    return jnp.mean(jnp.sum(nll_components, axis=-1))


# ----------------------------
# Lightweight pure-JAX Adam
# ----------------------------
def adam_init(params):
    m = jax.tree_util.tree_map(jnp.zeros_like, params)
    v = jax.tree_util.tree_map(jnp.zeros_like, params)
    t = jnp.array(0, dtype=jnp.int32)
    return {"m": m, "v": v, "t": t}


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1

    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"],
        grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"],
        grads,
    )

    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)

    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )

    return params_new, {"m": m_new, "v": v_new, "t": t_new}


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params,
        grads,
        opt_state,
        CONFIG["BASELINE_LR"],
    )
    return params, opt_state, loss


# ----------------------------
# Train baseline surrogate
# ----------------------------
key, subkey = jax.random.split(key)
sde_params = surrogate_model.init(subkey)
sde_opt_state = adam_init(sde_params)

baseline_losses = []
t_start = time.time()

for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, train_data, CONFIG["BASELINE_BATCH_SIZE"])

    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params,
        sde_opt_state,
        batch,
    )

    baseline_losses.append(float(loss))

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        train_loss = float(data_nll_loss(sde_params, train_data))
        test_loss = float(data_nll_loss(sde_params, test_data))
        print(
            f"[Baseline] epoch {epoch:5d} | "
            f"batch nll {float(loss): .5f} | "
            f"train nll {train_loss: .5f} | "
            f"test nll {test_loss: .5f}"
        )

print(f"Baseline training time: {time.time() - t_start:.2f} sec")

## Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library

In [ ]:
# ============================================================
# Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library
# ============================================================

# ------------------------------------------------------------------
# Each basis returns a 51-vector:
#     [tau, xi1, ..., xi50]
#
# Minimal 104-basis:
#   3 time-only generators:
#       ∂t, t∂t, t^2∂t
#   50 coordinate translations:
#       ∂x_i
#   50 coordinate scalings:
#       x_i ∂x_i
#   1 global Brownian/Gaeta scaling:
#       2(t/T)∂t + x·∇x
#
# Expanded Gaeta/low-rank additions:
#   1. Exponential translations:
#       e^{mu t/T} ∂x_i
#
#   2. Exponential projective linear-drift generators:
#       e^{2mu t/T}(∂t + mu x·∇x)
#
#   3. Low-rank rotations:
#       (v_b·x)v_a·∇x - (v_a·x)v_b·∇x
#
#   4. Low-rank shears:
#       (v_b·x)v_a·∇x
#
# The added basis elements are all at most linear in x, so xi_xx = 0.
# Therefore the Hessian-free determining-equation implementation remains valid.
# ------------------------------------------------------------------

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


def time_basis(t):
    """
    Normalized time variable.

    We use t/T rather than raw t inside exponentials and polynomial
    time basis functions for numerical stability.
    """
    return t / CONFIG["INPUT_T_SCALE"]


def safe_exp(z):
    c = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -c, c))


def zero_xi():
    return jnp.zeros((D,), dtype=jnp.float32)


def e_i(i):
    v = jnp.zeros((D,), dtype=jnp.float32)
    return v.at[i].set(1.0)


def unit(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


def make_low_rank_directions(K):
    """
    Generic low-rank directions in coordinate-index space.

    These are not fitted from the data and do not use the true SDE.
    They are fixed Fourier-like directions:
        constant, sin(1), cos(1), sin(2), cos(2), ...
    """
    idx = jnp.arange(D, dtype=jnp.float32)
    dirs = []

    # Global mean direction.
    dirs.append(unit(jnp.ones((D,), dtype=jnp.float32)))

    freq = 1
    while len(dirs) < K:
        phase = 2.0 * jnp.pi * freq * (idx + 0.5) / float(D)

        dirs.append(unit(jnp.sin(phase)))

        if len(dirs) < K:
            dirs.append(unit(jnp.cos(phase)))

        freq += 1

    return tuple(dirs[:K])


# ============================================================
# Group A: time-only projectable basis
# ============================================================

add_basis(
    "time_translation__dt",
    "Generic projectable time translation",
    lambda t, x: jnp.concatenate(
        [jnp.array([1.0], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_scaling__t_dt",
    "Generic projectable time scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t)], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_quadratic__t2_dt",
    "Generic projectable time quadratic",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t) ** 2], dtype=jnp.float32), zero_xi()]
    ),
)


# ============================================================
# Group B: coordinate translations ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"translation__dx{i+1}",
        "Generic coordinate translation",
        lambda t, x, i=i: jnp.concatenate(
            [jnp.array([0.0], dtype=jnp.float32), e_i(i)]
        ),
    )


# ============================================================
# Group C: coordinate scalings x_i ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"scaling__x{i+1}_dx{i+1}",
        "Generic coordinate scaling",
        lambda t, x, i=i: jnp.concatenate(
            [
                jnp.array([0.0], dtype=jnp.float32),
                jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
            ]
        ),
    )


# ============================================================
# Group D: global Brownian/Gaeta lifted scaling
# ============================================================

add_basis(
    "global_brownian_scaling__2t_dt_plus_x_dot_dx",
    "50D lift of Brownian/Gaeta scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([2.0 * time_basis(t)], dtype=jnp.float32), x]
    ),
)


# ============================================================
# Group E: Gaeta-style exponential linear-drift basis
# ============================================================

if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    for mu in CONFIG["BASIS_LINEAR_MU_LIST"]:
        mu = float(mu)

        if abs(mu) < 1e-12:
            continue

        # e^{mu t/T} ∂x_i
        for i in range(D):
            add_basis(
                f"exp_translation__mu_{mu:+.2f}__e_mut_dx{i+1}",
                "Gaeta-style exponential translation e^{mu t} ∂x_i",
                lambda t, x, mu=mu, i=i: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.zeros((D,), dtype=jnp.float32).at[i].set(
                            safe_exp(mu * time_basis(t))
                        ),
                    ]
                ),
            )

        # e^{2mu t/T}(∂t + mu x·∇x)
        add_basis(
            f"exp_projective__mu_{mu:+.2f}__e_2mut_dt_plus_mu_x_dot_dx",
            "Gaeta-style projective linear-drift generator",
            lambda t, x, mu=mu: jnp.concatenate(
                [
                    jnp.array(
                        [safe_exp(2.0 * mu * time_basis(t))],
                        dtype=jnp.float32,
                    ),
                    mu * safe_exp(2.0 * mu * time_basis(t)) * x,
                ]
            ),
        )


# ============================================================
# Group F: low-rank rotations and shears
# ============================================================

LOW_RANK_DIRECTIONS = make_low_rank_directions(CONFIG["BASIS_LOW_RANK_K"])
K_LOW = len(LOW_RANK_DIRECTIONS)

print(f"Using {K_LOW} fixed low-rank basis directions.")


# ----------------------------
# Low-rank rotations:
#   (v_b·x)v_a - (v_a·x)v_b
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    for a in range(K_LOW):
        for b in range(a + 1, K_LOW):
            va = LOW_RANK_DIRECTIONS[a]
            vb = LOW_RANK_DIRECTIONS[b]

            add_basis(
                f"lowrank_rotation__v{a+1}_v{b+1}",
                "Low-rank antisymmetric rotation generator",
                lambda t, x, va=va, vb=vb: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                    ]
                ),
            )


# ----------------------------
# Low-rank adjacent shears:
#   (v_b·x)v_a
#
# We use only adjacent directions and both orientations:
#   v_{a+1} -> v_a and v_a -> v_{a+1}
#
# This adds 2(K-1) shears instead of all K(K-1).
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    for a in range(K_LOW - 1):
        va = LOW_RANK_DIRECTIONS[a]
        vb = LOW_RANK_DIRECTIONS[a + 1]

        add_basis(
            f"lowrank_shear__v{a+2}_to_v{a+1}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(vb, x) * va,
                ]
            ),
        )

        add_basis(
            f"lowrank_shear__v{a+1}_to_v{a+2}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(va, x) * vb,
                ]
            ),
        )


# ============================================================
# Finalize basis library
# ============================================================

BASIS_NAMES = [b["name"] for b in BASIS_LIBRARY]
BASIS_SOURCES = [b["source"] for b in BASIS_LIBRARY]
BASIS_FNS = tuple(b["fn"] for b in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print(f"\nNumber of 50D basis functions: {N_BASIS}")
print(f"BASIS_MODE: {CONFIG['BASIS_MODE']}")

n_minimal = 3 + D + D + 1
n_exp = 0
if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    n_exp = len([mu for mu in CONFIG["BASIS_LINEAR_MU_LIST"] if abs(mu) > 1e-12]) * (D + 1)

n_rot = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    n_rot = K_LOW * (K_LOW - 1) // 2

n_shear = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    n_shear = 2 * (K_LOW - 1)

print("\nBasis count breakdown:")
print(f"  minimal time/translation/scaling/global : {n_minimal}")
print(f"  exponential Gaeta terms                 : {n_exp}")
print(f"  low-rank rotations                      : {n_rot}")
print(f"  low-rank shears                         : {n_shear}")
print(f"  total                                   : {N_BASIS}")

for i, (nm, src) in enumerate(zip(BASIS_NAMES, BASIS_SOURCES)):
    print(f"{i:03d} | {nm} | {src}")


def eval_basis_one(t, x):
    """
    Returns basis values at one point.

    Output shape:
        (R, 51)
    """
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    """
    Evaluate learned generators at one point.

    W shape:
        (m, R)

    z shape:
        (51,) = [t, x1, ..., x50]

    Returns:
        X shape (m, 51), with X[a] = [tau_a, xi_a1, ..., xi_a50].
    """
    t = z[0]
    x = z[1:]
    B = eval_basis_one(t, x)
    return W @ B


def eval_generators_batch(W, tx):
    """
    tx shape:
        (S, 51)

    Returns:
        X shape (S, m, 51)
    """
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


def _x_box_to_arrays(x_box):
    """
    Allows x_box to be either:
        (low_scalar, high_scalar)
    or:
        (low_vector_shape_D, high_vector_shape_D)
    """
    low, high = x_box

    low = jnp.asarray(low, dtype=jnp.float32)
    high = jnp.asarray(high, dtype=jnp.float32)

    if low.ndim == 0:
        low = jnp.full((D,), low, dtype=jnp.float32)

    if high.ndim == 0:
        high = jnp.full((D,), high, dtype=jnp.float32)

    if low.shape != (D,) or high.shape != (D,):
        raise ValueError(
            f"x_box low/high must be scalar or shape ({D},). "
            f"Got {low.shape}, {high.shape}."
        )

    return low, high


def sample_tx_box(key, n_points, t_box, x_box):
    """
    Sample collocation points in:
        [t_min,t_max] x x_box^50

    Output:
        tx shape (n_points, 51)
    """
    key_t, key_x = jax.random.split(key)

    t = jax.random.uniform(
        key_t,
        (n_points, 1),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )

    x_low, x_high = _x_box_to_arrays(x_box)

    u = jax.random.uniform(key_x, (n_points, D))
    x = x_low[None, :] + u * (x_high[None, :] - x_low[None, :])

    return jnp.concatenate([t, x], axis=1)

def sample_tx_data_jitter(key, dataset, n_points, jitter_std=0.05):
    """
    Sample symmetry collocation points near the observed trajectory manifold.
    """
    key_idx, key_noise = jax.random.split(key)

    t0, x0, _, _ = dataset
    n = t0.shape[0]

    idx = jax.random.randint(key_idx, (n_points,), 0, n)

    t = t0[idx]
    x = x0[idx]

    noise = jitter_std * jax.random.normal(key_noise, x.shape)
    x_jitter = x + noise

    return jnp.concatenate([t[:, None], x_jitter], axis=1)


key, subkey = jax.random.split(key)
W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    subkey,
    (CONFIG["N_GENERATORS"], N_BASIS),
)

print("\nInitial W shape:", W.shape)

## Cell 5 — 50D determining loss, global closure, discovery, and E2 W controls

In [ ]:
# ============================================================
# Cell 5 — 50D determining loss, global closure, discovery, and E2 W controls
# ============================================================

def det_residuals_point(sde_params, W_fixed, z):
    """Projectable Ito determining residuals for a diagonal 50D SDE."""
    f = f_vec_single(sde_params, z)
    sigma = sigma_vec_single(sde_params, z)

    J_f = jax.jacfwd(lambda zz: f_vec_single(sde_params, zz))(z)
    J_sigma = jax.jacfwd(lambda zz: sigma_vec_single(sde_params, zz))(z)

    f_t = J_f[:, 0]
    f_x = J_f[:, 1:]
    sigma_t = J_sigma[:, 0]
    sigma_x = J_sigma[:, 1:]

    gen_fun = lambda zz: eval_generators_point(W_fixed, zz)
    X = gen_fun(z)
    J = jax.jacfwd(gen_fun)(z)

    tau = X[:, 0]
    xi = X[:, 1:]
    tau_t = J[:, 0, 0]
    tau_x = J[:, 0, 1:]
    xi_t = J[:, 1:, 0]
    xi_x = J[:, 1:, 1:]

    adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_dot_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    time_f_tau = tau[:, None] * f_t[None, :] + tau_t[:, None] * f[None, :]

    # Every basis vector used by this notebook is at most linear in x,
    # so the Ito xi_xx term is identically zero.
    R_f = xi_t + adv_xi - xi_dot_grad_f - time_f_tau

    eye = jnp.eye(D, dtype=sigma.dtype)
    first_term = jnp.einsum("p,aip->aip", sigma, xi_x)
    diagonal_correction = (
        jnp.einsum("aj,ij->ai", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )
    R_sigma = first_term - diagonal_correction[:, :, None] * eye[None, :, :]

    return R_f, R_sigma, tau_x


def determining_loss(sde_params, W_fixed, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W_fixed, z)
    )(tx)
    det = jnp.mean(R_f ** 2) + jnp.mean(R_sigma ** 2)
    projectable = jnp.mean(tau_x ** 2)
    return det, projectable


def determining_loss_components(sde_params, W_fixed, tx):
    """Separate drift/diffusion components for final diagnostics."""
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W_fixed, z)
    )(tx)
    drift_det = jnp.mean(R_f ** 2)
    diffusion_det = jnp.mean(R_sigma ** 2)
    projectable = jnp.mean(tau_x ** 2)
    return drift_det, diffusion_det, projectable


def generator_jacobians_batch(W_fixed, tx):
    def one(z):
        gen_fun = lambda zz: eval_generators_point(W_fixed, zz)
        X = gen_fun(z)
        J = jax.jacfwd(gen_fun)(z)
        return X, J
    return jax.vmap(one)(tx)


def lie_closure_loss_global(W_fixed, tx):
    m = W_fixed.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W_fixed, tx)
    S = tx.shape[0]

    idx_i, idx_j = jnp.triu_indices(m, k=1)
    Xi = X[:, idx_i, :]
    Xj = X[:, idx_j, :]
    Ji = J[:, idx_i, :, :]
    Jj = J[:, idx_j, :, :]

    brackets = (
        jnp.einsum("spc,spqc->spq", Xi, Jj)
        - jnp.einsum("spc,spqc->spq", Xj, Ji)
    )

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(brackets.shape[1], -1).T

    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    C = jnp.linalg.solve(G + reg, V.T @ B)
    residual = B - V @ C
    return jnp.mean(residual ** 2)


def gram_independence_loss(W_fixed, tx):
    X = eval_generators_batch(W_fixed, tx)
    S, m, _ = X.shape
    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    G = (V.T @ V) / (Q * S)

    I = jnp.eye(m, dtype=G.dtype)
    loss_ind = jnp.mean((G - I) ** 2)
    diag = jnp.diag(G)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - diag) ** 2
    )
    return loss_ind, norm_hinge, G


def sym_discovery_objective(W_trainable, sde_params, tx):
    det, proj = determining_loss(sde_params, W_trainable, tx)
    close = lie_closure_loss_global(W_trainable, tx)
    ind, norm_hinge, _ = gram_independence_loss(W_trainable, tx)

    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * close
        + CONFIG["LAMBDA_IND"] * ind
        + CONFIG["LAMBDA_PROJECTABLE"] * proj
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )
    return total, {
        "total": total,
        "det": det,
        "close": close,
        "ind": ind,
        "projectable": proj,
        "norm_hinge": norm_hinge,
    }


sym_value_and_grad = jax.value_and_grad(
    lambda W_trainable, sde_params, tx:
        sym_discovery_objective(W_trainable, sde_params, tx)[0]
)


@jax.jit
def sym_train_step(W_trainable, opt_state, sde_params, tx):
    _, grads = sym_value_and_grad(W_trainable, sde_params, tx)
    W_trainable, opt_state = adam_update(
        W_trainable,
        grads,
        opt_state,
        CONFIG["SYM_DISCOVERY_LR"],
    )
    _, aux = sym_discovery_objective(W_trainable, sde_params, tx)
    return W_trainable, opt_state, aux


# ----------------------------
# Train learned generator weights
# ----------------------------
W_opt_state = adam_init(W)
sym_history = []
t_start = time.time()

for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    tx_disc = sample_tx_data_jitter(
        subkey,
        train_data,
        CONFIG["SYM_DISCOVERY_POINTS"],
        jitter_std=CONFIG["E2_JITTER_STD"],
    )

    W, W_opt_state, aux = sym_train_step(W, W_opt_state, sde_params, tx_disc)

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        aux_np = {k: float(v) for k, v in aux.items()}
        sym_history.append(aux_np)
        print(
            f"[SymDiscovery] epoch {epoch:5d} | "
            f"total {aux_np['total']: .4e} | "
            f"det {aux_np['det']: .4e} | "
            f"close {aux_np['close']: .4e} | "
            f"ind {aux_np['ind']: .4e} | "
            f"proj {aux_np['projectable']: .4e}"
        )

print(f"Symmetry discovery time: {time.time() - t_start:.2f} sec")

# Freeze a named copy for E2.
W_learned = jax.tree_util.tree_map(lambda x: jnp.array(x), W)


# ----------------------------
# Construct a fixed random-W control with matched empirical row norms
# ----------------------------
def empirical_generator_row_rms(W_fixed, tx):
    X = eval_generators_batch(W_fixed, tx)  # (S,m,Q)
    return jnp.sqrt(jnp.mean(X ** 2, axis=(0, 2)) + 1e-12)


key_ref = jax.random.PRNGKey(CONFIG["E2_RANDOM_W_SEED"])
key_ref, key_cloud = jax.random.split(key_ref)
W_random_raw = jax.random.normal(key_ref, W_learned.shape)

tx_w_norm = sample_tx_data_jitter(
    key_cloud,
    train_data,
    512,
    jitter_std=CONFIG["E2_JITTER_STD"],
)

learned_row_rms = empirical_generator_row_rms(W_learned, tx_w_norm)
random_row_rms = empirical_generator_row_rms(W_random_raw, tx_w_norm)
W_random = W_random_raw * (
    learned_row_rms / (random_row_rms + 1e-12)
)[:, None]
W_random = jax.lax.stop_gradient(W_random)

print("\nE2 generator controls")
print("  W_learned shape:", W_learned.shape)
print("  W_random shape :", W_random.shape)
print("  learned row-RMS range:",
      float(jnp.min(learned_row_rms)), "to", float(jnp.max(learned_row_rms)))
print("  random row-RMS range after matching:",
      float(jnp.min(empirical_generator_row_rms(W_random, tx_w_norm))), "to",
      float(jnp.max(empirical_generator_row_rms(W_random, tx_w_norm))))


## Cell 6 — E2 matched four-arm Stage-III experiment

In [ ]:
# ============================================================
# Cell 6 — E2 matched four-arm Stage-III experiment
# ============================================================

# Save the unique common Stage-I checkpoint.
sde_params_baseline = jax.tree_util.tree_map(lambda x: jnp.array(x), sde_params)


def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.sqrt(sum(jnp.sum(x ** 2) for x in leaves) + 1e-30)


def split_dataset_for_stage3_validation(dataset, validation_fraction):
    """Deterministic tail split used only to select E2 regularization weights."""
    t0, x0, dt, dx = dataset
    n = int(t0.shape[0])
    n_val = max(1, int(round(validation_fraction * n)))
    n_fit = n - n_val
    if n_fit < 1:
        raise ValueError("Stage-III validation split leaves no fitting transitions.")

    fit = (t0[:n_fit], x0[:n_fit], dt[:n_fit], dx[:n_fit])
    val = (t0[n_fit:], x0[n_fit:], dt[n_fit:], dx[n_fit:])
    return fit, val


stage3_fit_data, stage3_val_data = split_dataset_for_stage3_validation(
    train_data,
    CONFIG["E2_STAGE3_VALIDATION_FRACTION"],
)


def make_e2_schedule(seed, n_steps):
    """Store only deterministic PRNG keys; all arms reconstruct identical batches."""
    root = jax.random.PRNGKey(seed)
    keys = jax.random.split(root, 3 * n_steps).reshape(n_steps, 3, 2)
    return keys


def materialize_e2_step(schedule_keys, step, dataset):
    key_batch = schedule_keys[step, 0]
    key_extra = schedule_keys[step, 1]
    key_probe = schedule_keys[step, 2]

    batch = sample_batch(
        key_batch,
        dataset,
        CONFIG["E2_STAGE3_BATCH_SIZE"],
    )

    tx_extra = sample_tx_data_jitter(
        key_extra,
        dataset,
        CONFIG["E2_N_EXTRA_REG_POINTS"],
        jitter_std=CONFIG["E2_JITTER_STD"],
    )

    t_batch, x_batch, _, _ = batch
    n_data = min(CONFIG["E2_N_DATA_REG_POINTS"], int(t_batch.shape[0]))
    tx_data = jnp.concatenate(
        [t_batch[:n_data, None], x_batch[:n_data]],
        axis=1,
    )
    tx_reg = jnp.concatenate([tx_data, tx_extra], axis=0)

    # One independent Rademacher state-direction probe per collocation point.
    probes = jax.random.rademacher(
        key_probe,
        (tx_reg.shape[0], D),
        dtype=jnp.float32,
    )
    return batch, tx_reg, probes


# ----------------------------
# Generic Sobolev/Jacobian control
# ----------------------------
def sobolev_hutchinson_components(params, tx, probes):
    """
    Unbiased one-probe estimates of
        ||d f / d x||_F^2 / D
        ||d sigma / d x||_F^2 / D
    evaluated at the same tx collocation points as the symmetry residual.
    """
    def one(z, u):
        t = z[0]
        x = z[1:]

        def drift_of_x(xx):
            return surrogate_f_sigma(params, t, xx)[0]

        def sigma_of_x(xx):
            return surrogate_f_sigma(params, t, xx)[1]

        _, Jf_u = jax.jvp(drift_of_x, (x,), (u,))
        _, Js_u = jax.jvp(sigma_of_x, (x,), (u,))

        drift_term = jnp.sum(Jf_u ** 2) / float(D)
        sigma_term = jnp.sum(Js_u ** 2) / float(D)
        return drift_term, sigma_term

    drift_terms, sigma_terms = jax.vmap(one)(tx, probes)
    return jnp.mean(drift_terms), jnp.mean(sigma_terms)


def sobolev_hutchinson_loss(params, tx, probes):
    drift_sob, sigma_sob = sobolev_hutchinson_components(params, tx, probes)
    return drift_sob + sigma_sob


# ----------------------------
# Arm objectives
# Projectability is not included in Stage III because all library basis
# elements are projectable by construction. It remains a reported diagnostic.
# ----------------------------
def data_only_objective(params, batch):
    data = data_nll_loss(params, batch)
    return data, {
        "total": data,
        "data": data,
        "regularizer": jnp.array(0.0, dtype=data.dtype),
        "reg_drift": jnp.array(0.0, dtype=data.dtype),
        "reg_diffusion": jnp.array(0.0, dtype=data.dtype),
    }


def determining_arm_objective(params, W_fixed, batch, tx_reg, reg_lambda):
    data = data_nll_loss(params, batch)
    drift_det, diffusion_det, _ = determining_loss_components(
        params, W_fixed, tx_reg
    )
    reg = drift_det + diffusion_det
    total = data + reg_lambda * reg
    return total, {
        "total": total,
        "data": data,
        "regularizer": reg,
        "reg_drift": drift_det,
        "reg_diffusion": diffusion_det,
    }


def sobolev_arm_objective(params, batch, tx_reg, probes, reg_lambda):
    data = data_nll_loss(params, batch)
    drift_sob, diffusion_sob = sobolev_hutchinson_components(
        params, tx_reg, probes
    )
    reg = drift_sob + diffusion_sob
    total = data + reg_lambda * reg
    return total, {
        "total": total,
        "data": data,
        "regularizer": reg,
        "reg_drift": drift_sob,
        "reg_diffusion": diffusion_sob,
    }


@jax.jit
def data_only_step(params, opt_state, batch):
    (_, aux), grads = jax.value_and_grad(
        data_only_objective,
        has_aux=True,
    )(params, batch)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["E2_STAGE3_LR"]
    )
    return params, opt_state, aux


@jax.jit
def determining_step(params, opt_state, W_fixed, batch, tx_reg, reg_lambda):
    objective = lambda p: determining_arm_objective(
        p, W_fixed, batch, tx_reg, reg_lambda
    )
    (_, aux), grads = jax.value_and_grad(objective, has_aux=True)(params)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["E2_STAGE3_LR"]
    )
    return params, opt_state, aux


@jax.jit
def sobolev_step(params, opt_state, batch, tx_reg, probes, reg_lambda):
    objective = lambda p: sobolev_arm_objective(
        p, batch, tx_reg, probes, reg_lambda
    )
    (_, aux), grads = jax.value_and_grad(objective, has_aux=True)(params)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["E2_STAGE3_LR"]
    )
    return params, opt_state, aux


# ----------------------------
# Initial gradient matching
# ----------------------------
cal_schedule = make_e2_schedule(CONFIG["E2_CALIBRATION_SEED"], 1)
cal_batch, cal_tx, cal_probes = materialize_e2_step(
    cal_schedule, 0, stage3_fit_data
)

cal_data_grad = jax.grad(
    lambda p: data_nll_loss(p, cal_batch)
)(sde_params_baseline)
cal_data_grad_norm = tree_l2_norm(cal_data_grad)


def determining_reg_grad_and_value(W_fixed):
    value, grad = jax.value_and_grad(
        lambda p: determining_loss(p, W_fixed, cal_tx)[0]
    )(sde_params_baseline)
    return value, tree_l2_norm(grad)


def sobolev_reg_grad_and_value():
    value, grad = jax.value_and_grad(
        lambda p: sobolev_hutchinson_loss(p, cal_tx, cal_probes)
    )(sde_params_baseline)
    return value, tree_l2_norm(grad)


gl_reg0, gl_grad0 = determining_reg_grad_and_value(W_learned)
random_reg0, random_grad0 = determining_reg_grad_and_value(W_random)
sob_reg0, sob_grad0 = sobolev_reg_grad_and_value()

gl_reference_lambda = float(CONFIG["E2_GL_REFERENCE_LAMBDA"])
target_weighted_grad = gl_reference_lambda * float(gl_grad0)

base_lambda_random = target_weighted_grad / (float(random_grad0) + 1e-30)
base_lambda_sobolev = target_weighted_grad / (float(sob_grad0) + 1e-30)
base_lambda_gl = gl_reference_lambda

E2_BASE_LAMBDAS = {
    "random_w": float(base_lambda_random),
    "sobolev": float(base_lambda_sobolev),
    "gaeta_lie": float(base_lambda_gl),
}

print("\n" + "=" * 100)
print("E2 INITIAL GRADIENT MATCHING")
print("=" * 100)
print(f"data gradient norm                : {float(cal_data_grad_norm):.6e}")
print(f"learned-W raw regularizer         : {float(gl_reg0):.6e}")
print(f"learned-W unweighted grad norm    : {float(gl_grad0):.6e}")
print(f"reference GL lambda               : {gl_reference_lambda:.6e}")
print(f"target weighted grad norm         : {target_weighted_grad:.6e}")
print(f"random-W raw regularizer          : {float(random_reg0):.6e}")
print(f"random-W unweighted grad norm     : {float(random_grad0):.6e}")
print(f"random-W matched base lambda      : {base_lambda_random:.6e}")
print(f"Sobolev raw regularizer           : {float(sob_reg0):.6e}")
print(f"Sobolev unweighted grad norm      : {float(sob_grad0):.6e}")
print(f"Sobolev matched base lambda       : {base_lambda_sobolev:.6e}")


# ----------------------------
# Generic arm runner
# ----------------------------
def run_e2_arm(
    arm_name,
    reg_lambda,
    schedule,
    dataset,
    n_steps,
    print_progress=False,
):
    params = jax.tree_util.tree_map(lambda x: jnp.array(x), sde_params_baseline)
    opt_state = adam_init(params)
    history = []

    # Compile/warm up on the first scheduled update, then reset exactly.
    batch0, tx0, probes0 = materialize_e2_step(schedule, 0, dataset)
    if arm_name == "data_only":
        p_warm, s_warm, aux_warm = data_only_step(params, opt_state, batch0)
    elif arm_name == "random_w":
        p_warm, s_warm, aux_warm = determining_step(
            params, opt_state, W_random, batch0, tx0, reg_lambda
        )
    elif arm_name == "gaeta_lie":
        p_warm, s_warm, aux_warm = determining_step(
            params, opt_state, W_learned, batch0, tx0, reg_lambda
        )
    elif arm_name == "sobolev":
        p_warm, s_warm, aux_warm = sobolev_step(
            params, opt_state, batch0, tx0, probes0, reg_lambda
        )
    else:
        raise ValueError(f"Unknown E2 arm: {arm_name}")
    jax.block_until_ready(aux_warm["total"])

    params = jax.tree_util.tree_map(lambda x: jnp.array(x), sde_params_baseline)
    opt_state = adam_init(params)

    start = time.perf_counter()
    last_aux = None

    for step in range(n_steps):
        batch, tx_reg, probes = materialize_e2_step(schedule, step, dataset)

        if arm_name == "data_only":
            params, opt_state, aux = data_only_step(params, opt_state, batch)
        elif arm_name == "random_w":
            params, opt_state, aux = determining_step(
                params, opt_state, W_random, batch, tx_reg, reg_lambda
            )
        elif arm_name == "gaeta_lie":
            params, opt_state, aux = determining_step(
                params, opt_state, W_learned, batch, tx_reg, reg_lambda
            )
        else:
            params, opt_state, aux = sobolev_step(
                params, opt_state, batch, tx_reg, probes, reg_lambda
            )

        last_aux = aux

        if (
            print_progress
            and ((step + 1) % CONFIG["PRINT_EVERY"] == 0 or step == 0)
        ):
            jax.block_until_ready(aux["total"])
            aux_np = {k: float(v) for k, v in aux.items()}
            val_nll = float(data_nll_loss(params, stage3_val_data))
            history.append({
                "step": step + 1,
                "val_nll": val_nll,
                **aux_np,
            })
            print(
                f"[{arm_name}] step {step + 1:5d} | "
                f"total {aux_np['total']:.6e} | "
                f"data {aux_np['data']:.6e} | "
                f"reg {aux_np['regularizer']:.6e} | "
                f"val nll {val_nll:.6e}"
            )

    if last_aux is not None:
        jax.block_until_ready(last_aux["total"])
    total_sec = time.perf_counter() - start

    return params, history, {
        "total_sec": float(total_sec),
        "mean_step_ms": float(1000.0 * total_sec / n_steps),
        "n_updates": int(n_steps),
    }


# ----------------------------
# Tune each regularized arm using only Stage-III validation NLL
# ----------------------------
tuning_schedule = make_e2_schedule(
    CONFIG["E2_TUNING_SCHEDULE_SEED"],
    CONFIG["E2_TUNING_EPOCHS"],
)

e2_tuning_report = {}
e2_selected_lambdas = {"data_only": 0.0}

for arm_name in ("random_w", "sobolev", "gaeta_lie"):
    base_lambda = E2_BASE_LAMBDAS[arm_name]
    candidate_rows = []

    print("\n" + "-" * 100)
    print(f"Tuning E2 arm: {arm_name}")
    print("-" * 100)

    for multiplier in CONFIG["E2_WEIGHT_MULTIPLIERS"]:
        candidate_lambda = float(multiplier * base_lambda)
        candidate_params, _, candidate_runtime = run_e2_arm(
            arm_name=arm_name,
            reg_lambda=candidate_lambda,
            schedule=tuning_schedule,
            dataset=stage3_fit_data,
            n_steps=CONFIG["E2_TUNING_EPOCHS"],
            print_progress=False,
        )
        val_nll = float(data_nll_loss(candidate_params, stage3_val_data))
        candidate_rows.append({
            "multiplier": float(multiplier),
            "lambda": candidate_lambda,
            "validation_nll": val_nll,
            "tuning_runtime_sec": candidate_runtime["total_sec"],
        })
        print(
            f"  multiplier={multiplier:>4.1f} | "
            f"lambda={candidate_lambda:.6e} | "
            f"validation NLL={val_nll:.6e}"
        )

    best = min(candidate_rows, key=lambda row: row["validation_nll"])
    e2_tuning_report[arm_name] = candidate_rows
    e2_selected_lambdas[arm_name] = float(best["lambda"])
    print(
        f"SELECTED {arm_name}: multiplier={best['multiplier']:.1f}, "
        f"lambda={best['lambda']:.6e}, "
        f"validation NLL={best['validation_nll']:.6e}"
    )


# ----------------------------
# Initial-gradient report at selected weights
# ----------------------------
e2_gradient_match_report = {
    "random_w": {
        "raw_regularizer": float(random_reg0),
        "regularizer_grad_norm": float(random_grad0),
        "selected_lambda": e2_selected_lambdas["random_w"],
        "weighted_grad_norm": e2_selected_lambdas["random_w"] * float(random_grad0),
        "data_grad_norm": float(cal_data_grad_norm),
        "weighted_to_data_ratio": (
            e2_selected_lambdas["random_w"] * float(random_grad0)
            / (float(cal_data_grad_norm) + 1e-30)
        ),
    },
    "sobolev": {
        "raw_regularizer": float(sob_reg0),
        "regularizer_grad_norm": float(sob_grad0),
        "selected_lambda": e2_selected_lambdas["sobolev"],
        "weighted_grad_norm": e2_selected_lambdas["sobolev"] * float(sob_grad0),
        "data_grad_norm": float(cal_data_grad_norm),
        "weighted_to_data_ratio": (
            e2_selected_lambdas["sobolev"] * float(sob_grad0)
            / (float(cal_data_grad_norm) + 1e-30)
        ),
    },
    "gaeta_lie": {
        "raw_regularizer": float(gl_reg0),
        "regularizer_grad_norm": float(gl_grad0),
        "selected_lambda": e2_selected_lambdas["gaeta_lie"],
        "weighted_grad_norm": e2_selected_lambdas["gaeta_lie"] * float(gl_grad0),
        "data_grad_norm": float(cal_data_grad_norm),
        "weighted_to_data_ratio": (
            e2_selected_lambdas["gaeta_lie"] * float(gl_grad0)
            / (float(cal_data_grad_norm) + 1e-30)
        ),
    },
}


# ----------------------------
# Final matched training from the identical Stage-I checkpoint
# ----------------------------
final_schedule = make_e2_schedule(
    CONFIG["E2_FINAL_SCHEDULE_SEED"],
    CONFIG["E2_STAGE3_EPOCHS"],
)

e2_arm_histories = {}
e2_runtime_report = {
    "paired_schedule_used": True,
    "weight_selection_uses_ground_truth_mse": False,
    "weight_selection_uses_original_test_set": False,
}

final_params = {}

for arm_name in CONFIG["E2_ARMS"]:
    print("\n" + "=" * 100)
    print(f"FINAL MATCHED E2 TRAINING ARM: {arm_name}")
    print("=" * 100)

    params_arm, history_arm, runtime_arm = run_e2_arm(
        arm_name=arm_name,
        reg_lambda=e2_selected_lambdas[arm_name],
        schedule=final_schedule,
        dataset=train_data,
        n_steps=CONFIG["E2_STAGE3_EPOCHS"],
        print_progress=True,
    )

    final_params[arm_name] = params_arm
    e2_arm_histories[arm_name] = history_arm
    e2_runtime_report[arm_name] = runtime_arm


sde_params_data_only = final_params["data_only"]
sde_params_random_w = final_params["random_w"]
sde_params_sobolev = final_params["sobolev"]
sde_params_gaeta_lie = final_params["gaeta_lie"]

# Compatibility aliases for any later exploratory notebook code.
sde_params_refined = sde_params_gaeta_lie
stage_f_history = e2_arm_histories["gaeta_lie"]

print("\nE2 final-arm training complete.")
for arm_name in CONFIG["E2_ARMS"]:
    print(
        f"  {arm_name:<10} | lambda={e2_selected_lambdas[arm_name]:.6e} | "
        f"runtime={e2_runtime_report[arm_name]['total_sec']:.2f}s | "
        f"mean step={e2_runtime_report[arm_name]['mean_step_ms']:.2f}ms"
    )


## Cell 7 — Final automatic E2 evaluation and machine-readable output

In [ ]:
# ============================================================
# Cell 7 — Final automatic E2 evaluation and machine-readable output
# ============================================================

E2_MODELS = {
    "stage1_baseline": sde_params_baseline,
    "data_only": sde_params_data_only,
    "random_w": sde_params_random_w,
    "sobolev": sde_params_sobolev,
    "gaeta_lie": sde_params_gaeta_lie,
}

E2_ARM_ORDER = [
    "stage1_baseline",
    "data_only",
    "random_w",
    "sobolev",
    "gaeta_lie",
]

E2_DISPLAY_NAMES = {
    "stage1_baseline": "Stage-I baseline",
    "data_only": "Data-only continuation",
    "random_w": "Random-W control",
    "sobolev": "Sobolev control",
    "gaeta_lie": "Gaeta-Lie",
}


def safe_float(x):
    return float(np.asarray(jax.device_get(x)))


def safe_pct_reduction(reference, value):
    return 100.0 * (float(reference) - float(value)) / (abs(float(reference)) + 1e-12)


def tree_all_finite(tree):
    return all(
        np.all(np.isfinite(np.asarray(jax.device_get(x))))
        for x in jax.tree_util.tree_leaves(tree)
    )


def coefficient_metrics(params, tx_eval):
    t_eval = tx_eval[:, 0]
    x_eval = tx_eval[:, 1:]

    drift_pred, sigma_pred = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(t_eval, x_eval)
    drift_true = jax.vmap(true_drift)(t_eval, x_eval)
    sigma_true = jax.vmap(true_sigma)(t_eval, x_eval)

    drift_error = drift_pred - drift_true
    sigma_error = sigma_pred - sigma_true

    drift_mse = jnp.mean(drift_error ** 2)
    sigma_mse = jnp.mean(sigma_error ** 2)

    return {
        "drift_mse": safe_float(drift_mse),
        "sigma_mse": safe_float(sigma_mse),
        "drift_rmse": safe_float(jnp.sqrt(drift_mse)),
        "sigma_rmse": safe_float(jnp.sqrt(sigma_mse)),
        "drift_mae": safe_float(jnp.mean(jnp.abs(drift_error))),
        "sigma_mae": safe_float(jnp.mean(jnp.abs(sigma_error))),
        "drift_relative_l2": safe_float(
            jnp.linalg.norm(drift_error) / (jnp.linalg.norm(drift_true) + 1e-12)
        ),
        "sigma_relative_l2": safe_float(
            jnp.linalg.norm(sigma_error) / (jnp.linalg.norm(sigma_true) + 1e-12)
        ),
    }


def exact_sobolev_components_single(params, z):
    t = z[0]
    x = z[1:]

    Jf = jax.jacfwd(lambda xx: surrogate_f_sigma(params, t, xx)[0])(x)
    Js = jax.jacfwd(lambda xx: surrogate_f_sigma(params, t, xx)[1])(x)

    return (
        jnp.sum(Jf ** 2) / float(D),
        jnp.sum(Js ** 2) / float(D),
    )


def exact_sobolev_components_batch(params, tx):
    drift_terms, sigma_terms = jax.vmap(
        lambda z: exact_sobolev_components_single(params, z)
    )(tx)
    drift_mean = jnp.mean(drift_terms)
    sigma_mean = jnp.mean(sigma_terms)
    return drift_mean, sigma_mean, drift_mean + sigma_mean


def determining_metrics(params, W_fixed, tx):
    drift_det, diffusion_det, projectable = determining_loss_components(
        params, W_fixed, tx
    )
    return {
        "drift": safe_float(drift_det),
        "diffusion": safe_float(diffusion_det),
        "total": safe_float(drift_det + diffusion_det),
        "projectable": safe_float(projectable),
    }


def one_step_metrics(params, dataset):
    t0, x0, dt, dx = dataset
    f, sigma = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(t0, x0)

    mean_dx = f * dt[:, None]
    var_dx = sigma ** 2 * dt[:, None] + 1e-8
    residual = dx - mean_dx

    return {
        "increment_rmse": safe_float(jnp.sqrt(jnp.mean(residual ** 2))),
        "standardized_sq_residual_mean": safe_float(
            jnp.mean(residual ** 2 / var_dx)
        ),
    }


# ----------------------------
# Frozen common evaluation clouds
# ----------------------------
eval_key = jax.random.PRNGKey(CONFIG["E2_EVAL_SEED"])
key_coeff, key_train, key_jitter = jax.random.split(eval_key, 3)

# Match the active system's time and state support after Cell 2 updates CONFIG.
tx_coeff_eval = sample_tx_box(
    key_coeff,
    CONFIG["E2_N_COEFF_EVAL_POINTS"],
    (0.0, CONFIG["T_FINAL"]),
    CONFIG["STAGE_F_EXTRA_X_BOX"] if "STAGE_F_EXTRA_X_BOX" in CONFIG
    else CONFIG["E2_COEFF_EVAL_X_BOX"],
)

batch_resid = sample_batch(
    key_train,
    train_data,
    CONFIG["E2_N_RESIDUAL_EVAL_POINTS"],
)
t_resid, x_resid, _, _ = batch_resid
tx_train_eval = jnp.concatenate([t_resid[:, None], x_resid], axis=1)

tx_jitter_eval = sample_tx_data_jitter(
    key_jitter,
    train_data,
    CONFIG["E2_N_RESIDUAL_EVAL_POINTS"],
    jitter_std=CONFIG["E2_JITTER_STD"],
)

tx_exact_sob = tx_jitter_eval[: CONFIG["E2_N_EXACT_SOB_EVAL_POINTS"]]


# ----------------------------
# Evaluate all five reported models
# ----------------------------
e2_results = {}

for arm_name in E2_ARM_ORDER:
    params = E2_MODELS[arm_name]
    if not tree_all_finite(params):
        raise FloatingPointError(f"Non-finite parameters in E2 arm: {arm_name}")

    coeff = coefficient_metrics(params, tx_coeff_eval)
    train_nll = safe_float(data_nll_loss(params, train_data))
    test_nll = safe_float(data_nll_loss(params, test_data))
    train_step = one_step_metrics(params, train_data)
    test_step = one_step_metrics(params, test_data)

    learned_train = determining_metrics(params, W_learned, tx_train_eval)
    learned_jitter = determining_metrics(params, W_learned, tx_jitter_eval)
    random_train = determining_metrics(params, W_random, tx_train_eval)
    random_jitter = determining_metrics(params, W_random, tx_jitter_eval)

    sob_drift, sob_sigma, sob_total = exact_sobolev_components_batch(
        params, tx_exact_sob
    )

    e2_results[arm_name] = {
        **coeff,
        "train_nll": train_nll,
        "test_nll": test_nll,
        "train_increment_rmse": train_step["increment_rmse"],
        "test_increment_rmse": test_step["increment_rmse"],
        "train_standardized_sq_residual": train_step[
            "standardized_sq_residual_mean"
        ],
        "test_standardized_sq_residual": test_step[
            "standardized_sq_residual_mean"
        ],
        "learned_w_det_train_drift": learned_train["drift"],
        "learned_w_det_train_diffusion": learned_train["diffusion"],
        "learned_w_det_train_total": learned_train["total"],
        "learned_w_det_jitter_drift": learned_jitter["drift"],
        "learned_w_det_jitter_diffusion": learned_jitter["diffusion"],
        "learned_w_det_jitter_total": learned_jitter["total"],
        "learned_w_projectability_jitter": learned_jitter["projectable"],
        "random_w_det_train_total": random_train["total"],
        "random_w_det_jitter_total": random_jitter["total"],
        "sobolev_drift_exact": safe_float(sob_drift),
        "sobolev_sigma_exact": safe_float(sob_sigma),
        "sobolev_total_exact": safe_float(sob_total),
    }


baseline = e2_results["stage1_baseline"]
data_only = e2_results["data_only"]
sobolev = e2_results["sobolev"]
gaeta = e2_results["gaeta_lie"]

for arm_name in E2_ARM_ORDER:
    r = e2_results[arm_name]
    r["drift_improvement_vs_stage1_pct"] = safe_pct_reduction(
        baseline["drift_mse"], r["drift_mse"]
    )
    r["sigma_improvement_vs_stage1_pct"] = safe_pct_reduction(
        baseline["sigma_mse"], r["sigma_mse"]
    )
    r["drift_improvement_vs_data_only_pct"] = safe_pct_reduction(
        data_only["drift_mse"], r["drift_mse"]
    )
    r["sigma_improvement_vs_data_only_pct"] = safe_pct_reduction(
        data_only["sigma_mse"], r["sigma_mse"]
    )
    r["selected_lambda"] = float(e2_selected_lambdas.get(arm_name, 0.0))

    runtime = e2_runtime_report.get(arm_name, {})
    r["runtime_sec"] = float(runtime.get("total_sec", np.nan))
    r["mean_step_ms"] = float(runtime.get("mean_step_ms", np.nan))
    r["n_updates"] = int(runtime.get("n_updates", 0))


gaeta_vs_sobolev = {
    "drift_advantage_pct": safe_pct_reduction(
        sobolev["drift_mse"], gaeta["drift_mse"]
    ),
    "sigma_advantage_pct": safe_pct_reduction(
        sobolev["sigma_mse"], gaeta["sigma_mse"]
    ),
    "test_nll_advantage_pct": safe_pct_reduction(
        sobolev["test_nll"], gaeta["test_nll"]
    ),
}


# ----------------------------
# Fairness audit
# ----------------------------
final_update_counts = [
    e2_runtime_report[name]["n_updates"] for name in CONFIG["E2_ARMS"]
]

fairness_report = {
    "same_stage1_checkpoint": True,
    "same_final_update_count": len(set(final_update_counts)) == 1,
    "same_final_data_batch_schedule": bool(
        e2_runtime_report.get("paired_schedule_used", False)
    ),
    "same_final_collocation_schedule": bool(
        e2_runtime_report.get("paired_schedule_used", False)
    ),
    "ground_truth_mse_used_for_weight_selection": bool(
        e2_runtime_report.get(
            "weight_selection_uses_ground_truth_mse", False
        )
    ),
    "original_test_set_used_for_weight_selection": bool(
        e2_runtime_report.get(
            "weight_selection_uses_original_test_set", False
        )
    ),
}

if fairness_report["ground_truth_mse_used_for_weight_selection"]:
    raise RuntimeError("Invalid E2 run: ground-truth MSE was used for tuning.")
if fairness_report["original_test_set_used_for_weight_selection"]:
    raise RuntimeError("Invalid E2 run: the original test set was used for tuning.")


# ----------------------------
# Human-readable complete report
# ----------------------------
print("\n" + "=" * 118)
print("E2 — MATCHED SOBOLEV / SMOOTHNESS CONTROL: COMPLETE FINAL REPORT")
print("=" * 118)
print(f"SDE                              : {CONFIG['TRUE_SDE_NAME']}")
print(f"Description                      : {ACTIVE_SDE['description']}")
print(f"State dimension                  : {D}")
print(f"Run seed                         : {CONFIG['SEED']}")
print(f"Number of learned generators     : {CONFIG['N_GENERATORS']}")
print(f"Number of basis templates        : {N_BASIS}")
print(f"Final Stage-III updates per arm  : {CONFIG['E2_STAGE3_EPOCHS']}")
print(f"Final Stage-III batch size       : {CONFIG['E2_STAGE3_BATCH_SIZE']}")
print(f"Trajectory reg points/update     : {CONFIG['E2_N_DATA_REG_POINTS']}")
print(f"Extra jitter reg points/update   : {CONFIG['E2_N_EXTRA_REG_POINTS']}")
print(f"Jitter standard deviation        : {CONFIG['E2_JITTER_STD']}")
print(f"Coefficient evaluation points    : {tx_coeff_eval.shape[0]}")
print(f"Residual evaluation points       : {tx_jitter_eval.shape[0]}")
print(f"Exact Sobolev evaluation points  : {tx_exact_sob.shape[0]}")

print("\nFAIRNESS AUDIT")
for name, value in fairness_report.items():
    expected = False if name in (
        "ground_truth_mse_used_for_weight_selection",
        "original_test_set_used_for_weight_selection",
    ) else True
    status = "PASS" if value == expected else "FAIL"
    print(f"  [{status}] {name}: {value}")

print("\nWEIGHT-SELECTION RESULTS")
for arm_name in ("random_w", "sobolev", "gaeta_lie"):
    print(f"\n  {E2_DISPLAY_NAMES[arm_name]}")
    for row in e2_tuning_report[arm_name]:
        selected = abs(row["lambda"] - e2_selected_lambdas[arm_name]) <= (
            1e-12 * max(1.0, abs(row["lambda"]))
        )
        mark = "SELECTED" if selected else ""
        print(
            f"    multiplier={row['multiplier']:.1f} | "
            f"lambda={row['lambda']:.6e} | "
            f"validation NLL={row['validation_nll']:.6e} {mark}"
        )

print("\nINITIAL REGULARIZER / GRADIENT SCALE")
for arm_name in ("random_w", "sobolev", "gaeta_lie"):
    r = e2_gradient_match_report[arm_name]
    print(f"\n  {E2_DISPLAY_NAMES[arm_name]}")
    print(f"    raw regularizer               : {r['raw_regularizer']:.6e}")
    print(f"    unweighted regularizer |grad| : {r['regularizer_grad_norm']:.6e}")
    print(f"    selected lambda               : {r['selected_lambda']:.6e}")
    print(f"    weighted regularizer |grad|   : {r['weighted_grad_norm']:.6e}")
    print(f"    data-loss |grad|              : {r['data_grad_norm']:.6e}")
    print(f"    weighted/data gradient ratio  : {r['weighted_to_data_ratio']:.6e}")

header = (
    f"{'ARM':<25}"
    f"{'DRIFT MSE':>14}"
    f"{'SIGMA MSE':>14}"
    f"{'TEST NLL':>14}"
    f"{'DRIFT Δ%':>12}"
    f"{'SIGMA Δ%':>12}"
    f"{'TIME(s)':>11}"
    f"{'ms/step':>11}"
)
print("\nPRIMARY E2 RESULTS")
print(header)
print("-" * len(header))
for arm_name in E2_ARM_ORDER:
    r = e2_results[arm_name]
    print(
        f"{E2_DISPLAY_NAMES[arm_name]:<25}"
        f"{r['drift_mse']:>14.6e}"
        f"{r['sigma_mse']:>14.6e}"
        f"{r['test_nll']:>14.6e}"
        f"{r['drift_improvement_vs_stage1_pct']:>12.2f}"
        f"{r['sigma_improvement_vs_stage1_pct']:>12.2f}"
        f"{r['runtime_sec']:>11.2f}"
        f"{r['mean_step_ms']:>11.2f}"
    )

print("\nSECONDARY COEFFICIENT AND DATA-FIT METRICS")
for arm_name in E2_ARM_ORDER:
    r = e2_results[arm_name]
    print(f"\n  {E2_DISPLAY_NAMES[arm_name]}")
    print(f"    drift RMSE                         : {r['drift_rmse']:.6e}")
    print(f"    sigma RMSE                         : {r['sigma_rmse']:.6e}")
    print(f"    drift MAE                          : {r['drift_mae']:.6e}")
    print(f"    sigma MAE                          : {r['sigma_mae']:.6e}")
    print(f"    drift relative L2                  : {r['drift_relative_l2']:.6e}")
    print(f"    sigma relative L2                  : {r['sigma_relative_l2']:.6e}")
    print(f"    train NLL                          : {r['train_nll']:.6e}")
    print(f"    test NLL                           : {r['test_nll']:.6e}")
    print(f"    train increment RMSE               : {r['train_increment_rmse']:.6e}")
    print(f"    test increment RMSE                : {r['test_increment_rmse']:.6e}")
    print(f"    train standardized sq residual     : {r['train_standardized_sq_residual']:.6e}")
    print(f"    test standardized sq residual      : {r['test_standardized_sq_residual']:.6e}")

print("\nMECHANISM DIAGNOSTICS")
for arm_name in E2_ARM_ORDER:
    r = e2_results[arm_name]
    print(f"\n  {E2_DISPLAY_NAMES[arm_name]}")
    print(f"    learned-W train drift residual     : {r['learned_w_det_train_drift']:.6e}")
    print(f"    learned-W train diffusion residual : {r['learned_w_det_train_diffusion']:.6e}")
    print(f"    learned-W train total residual     : {r['learned_w_det_train_total']:.6e}")
    print(f"    learned-W jitter drift residual    : {r['learned_w_det_jitter_drift']:.6e}")
    print(f"    learned-W jitter diffusion residual: {r['learned_w_det_jitter_diffusion']:.6e}")
    print(f"    learned-W jitter total residual    : {r['learned_w_det_jitter_total']:.6e}")
    print(f"    random-W train total residual      : {r['random_w_det_train_total']:.6e}")
    print(f"    random-W jitter total residual     : {r['random_w_det_jitter_total']:.6e}")
    print(f"    exact drift Sobolev penalty        : {r['sobolev_drift_exact']:.6e}")
    print(f"    exact sigma Sobolev penalty        : {r['sobolev_sigma_exact']:.6e}")
    print(f"    exact total Sobolev penalty        : {r['sobolev_total_exact']:.6e}")

print("\nDIRECT GAETA-LIE VERSUS SOBOLEV")
print(
    f"  Gaeta-Lie drift-MSE advantage over Sobolev: "
    f"{gaeta_vs_sobolev['drift_advantage_pct']:.2f}%"
)
print(
    f"  Gaeta-Lie sigma-MSE advantage over Sobolev: "
    f"{gaeta_vs_sobolev['sigma_advantage_pct']:.2f}%"
)
print(
    f"  Gaeta-Lie test-NLL advantage over Sobolev : "
    f"{gaeta_vs_sobolev['test_nll_advantage_pct']:.2f}%"
)

print("\nADDITIONAL-OPTIMIZATION-CONTROLLED COMPARISON")
print(
    f"  Gaeta-Lie drift improvement vs data-only : "
    f"{gaeta['drift_improvement_vs_data_only_pct']:.2f}%"
)
print(
    f"  Gaeta-Lie sigma improvement vs data-only : "
    f"{gaeta['sigma_improvement_vs_data_only_pct']:.2f}%"
)
print(
    f"  Sobolev drift improvement vs data-only   : "
    f"{sobolev['drift_improvement_vs_data_only_pct']:.2f}%"
)
print(
    f"  Sobolev sigma improvement vs data-only   : "
    f"{sobolev['sigma_improvement_vs_data_only_pct']:.2f}%"
)


# ----------------------------
# Automatic single-run verdict
# ----------------------------
tol = CONFIG["E2_MATERIAL_ADVANTAGE_PCT"]
if (
    gaeta_vs_sobolev["drift_advantage_pct"] > tol
    and gaeta_vs_sobolev["sigma_advantage_pct"] > tol
    and gaeta["drift_improvement_vs_data_only_pct"] > 0.0
    and gaeta["sigma_improvement_vs_data_only_pct"] > 0.0
):
    e2_single_run_verdict = (
        "SUPPORTIVE: Gaeta-Lie materially beats generic Sobolev smoothing "
        "and data-only continuation on both coefficient metrics."
    )
elif (
    gaeta["drift_mse"] < sobolev["drift_mse"]
    and gaeta["sigma_mse"] < sobolev["sigma_mse"]
):
    e2_single_run_verdict = (
        "WEAKLY SUPPORTIVE: Gaeta-Lie is best on both coefficient metrics, "
        "but the single-run margin is smaller than the materiality threshold."
    )
elif (
    sobolev["drift_mse"] <= gaeta["drift_mse"]
    and sobolev["sigma_mse"] <= gaeta["sigma_mse"]
):
    e2_single_run_verdict = (
        "NOT SYMMETRY-SPECIFIC IN THIS RUN: generic Sobolev smoothing "
        "matches or outperforms Gaeta-Lie on both coefficient metrics."
    )
else:
    e2_single_run_verdict = (
        "MIXED: Gaeta-Lie and Sobolev split drift and diffusion. "
        "Use paired aggregation across systems and seeds before concluding."
    )

print("\nSINGLE-RUN E2 VERDICT")
print("  " + e2_single_run_verdict)
print(
    "  Conclusions should be based on paired results across the "
    "three 50D systems and preferably five seeds."
)


# ----------------------------
# Save long-form CSV and JSON
# ----------------------------
long_rows = []
for arm_name in E2_ARM_ORDER:
    long_rows.append({
        "experiment": "E2_matched_sobolev_control",
        "sde_name": CONFIG["TRUE_SDE_NAME"],
        "state_dim": int(D),
        "seed": int(CONFIG["SEED"]),
        "arm": arm_name,
        "arm_display_name": E2_DISPLAY_NAMES[arm_name],
        "stage3_epochs": int(CONFIG["E2_STAGE3_EPOCHS"]),
        "stage3_batch_size": int(CONFIG["E2_STAGE3_BATCH_SIZE"]),
        "n_data_reg_points": int(CONFIG["E2_N_DATA_REG_POINTS"]),
        "n_extra_reg_points": int(CONFIG["E2_N_EXTRA_REG_POINTS"]),
        "collocation_type": "trajectory_plus_gaussian_jitter",
        "jitter_std": float(CONFIG["E2_JITTER_STD"]),
        **e2_results[arm_name],
    })

csv_name = (
    f"E2_{CONFIG['TRUE_SDE_NAME']}_seed{CONFIG['SEED']}_long.csv"
)
json_name = (
    f"E2_{CONFIG['TRUE_SDE_NAME']}_seed{CONFIG['SEED']}_summary.json"
)

with open(csv_name, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(long_rows[0].keys()))
    writer.writeheader()
    writer.writerows(long_rows)

summary = {
    "experiment": "E2_matched_sobolev_control",
    "sde_name": CONFIG["TRUE_SDE_NAME"],
    "state_dim": int(D),
    "seed": int(CONFIG["SEED"]),
    "fairness_report": fairness_report,
    "base_lambdas": E2_BASE_LAMBDAS,
    "selected_lambdas": e2_selected_lambdas,
    "tuning_report": e2_tuning_report,
    "gradient_match_report": e2_gradient_match_report,
    "runtime_report": e2_runtime_report,
    "gaeta_lie_vs_sobolev": gaeta_vs_sobolev,
    "single_run_verdict": e2_single_run_verdict,
    "results": e2_results,
}

with open(json_name, "w") as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print("\nMACHINE-READABLE OUTPUT")
print(f"  CSV : {os.path.abspath(csv_name)}")
print(f"  JSON: {os.path.abspath(json_name)}")

print("\nE2_LONG_FORM_CSV")
csv_buffer = io.StringIO()
csv_writer = csv.DictWriter(
    csv_buffer,
    fieldnames=list(long_rows[0].keys()),
)
csv_writer.writeheader()
csv_writer.writerows(long_rows)
print(csv_buffer.getvalue().strip())

print("\n" + "=" * 118)
print("END OF E2 COMPLETE FINAL REPORT")
print("=" * 118)
